# RetainIQ — Phase 9.2: Chunking & Embedding Index

## Objective

I convert the knowledge corpus into retrieval-ready chunks and build a semantic embedding index when available, with a transparent TF-IDF fallback. I save the chunk metadata, vectors, and index configuration as reusable artifacts.

### Notebook presentation rule

For every relevant analytical code cell, I place a **Result & conclusion** markdown cell immediately after it. The analytical code also prints a concise `Result:` line based on the executed data. Setup-only cells are not padded with artificial conclusions.

In [3]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
from IPython.display import display

ROOT = Path('..')
OUTPUT_DIR = ROOT / 'outputs'
CONFIG_DIR = ROOT / 'config'

parameters = pd.read_csv(CONFIG_DIR / 'rag_parameters.csv')
param = dict(zip(parameters['parameter'], parameters['value']))
chunk_size = int(param['chunk_size'])
chunk_overlap = int(param['chunk_overlap'])
embedding_model_name = str(param['embedding_model'])

def print_result(message):
    print(f'Result: {message}')

In [4]:
knowledge_documents = pd.read_csv(OUTPUT_DIR / 'knowledge_documents.csv')
knowledge_documents['text'] = knowledge_documents['text'].fillna('').astype(str)
knowledge_documents = knowledge_documents[knowledge_documents['text'].str.len() > 0].copy()

print_result(f'Loaded {len(knowledge_documents):,} non-empty knowledge records for chunking.')
display(knowledge_documents[['document_id','topic','source_name','text']].head(5))

Result: Loaded 8,411 non-empty knowledge records for chunking.


,document_id,topic,source_name,text
0,vw_retention_strategy_priority:0,retention_strategy,vw_retention_strategy_priority,Source: vw_retention_strategy_priority | Topic...
1,vw_retention_strategy_priority:1,retention_strategy,vw_retention_strategy_priority,Source: vw_retention_strategy_priority | Topic...
2,vw_retention_strategy_priority:2,retention_strategy,vw_retention_strategy_priority,Source: vw_retention_strategy_priority | Topic...
3,vw_retention_strategy_priority:3,retention_strategy,vw_retention_strategy_priority,Source: vw_retention_strategy_priority | Topic...
4,vw_retention_strategy_priority:4,retention_strategy,vw_retention_strategy_priority,Source: vw_retention_strategy_priority | Topic...


### Result & conclusion

The knowledge records are ready for chunking. Empty records are excluded because they cannot provide retrievable evidence.

In [5]:
def normalize_text(text):
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def chunk_text(text, size=900, overlap=150):
    text = normalize_text(text)
    if len(text) <= size:
        return [text]
    chunks=[]
    start=0
    step=max(1, size-overlap)
    while start < len(text):
        chunk=text[start:start+size].strip()
        if chunk:
            chunks.append(chunk)
        if start + size >= len(text):
            break
        start += step
    return chunks

chunk_records=[]
for row in knowledge_documents.itertuples(index=False):
    chunks = chunk_text(row.text, chunk_size, chunk_overlap)
    for chunk_number, chunk in enumerate(chunks):
        chunk_records.append({
            'chunk_id': f'{row.document_id}:chunk_{chunk_number}',
            'document_id': row.document_id,
            'source_name': row.source_name,
            'source_type': row.source_type,
            'topic': row.topic,
            'chunk_number': chunk_number,
            'chunk_text': chunk,
        })

chunks_df = pd.DataFrame(chunk_records)
chunks_df.to_csv(OUTPUT_DIR / 'retrieval_chunks.csv', index=False)
print_result(f'Created {len(chunks_df):,} retrieval chunks from {len(knowledge_documents):,} source records.')
display(chunks_df[['chunk_id','topic','chunk_text']].head(8))

Result: Created 8,414 retrieval chunks from 8,411 source records.


,chunk_id,topic,chunk_text
0,vw_retention_strategy_priority:0:chunk_0,retention_strategy,Source: vw_retention_strategy_priority | Topic...
1,vw_retention_strategy_priority:1:chunk_0,retention_strategy,Source: vw_retention_strategy_priority | Topic...
2,vw_retention_strategy_priority:2:chunk_0,retention_strategy,Source: vw_retention_strategy_priority | Topic...
3,vw_retention_strategy_priority:3:chunk_0,retention_strategy,Source: vw_retention_strategy_priority | Topic...
4,vw_retention_strategy_priority:4:chunk_0,retention_strategy,Source: vw_retention_strategy_priority | Topic...
5,vw_retention_strategy_priority:5:chunk_0,retention_strategy,Source: vw_retention_strategy_priority | Topic...
6,vw_retention_strategy_priority:6:chunk_0,retention_strategy,Source: vw_retention_strategy_priority | Topic...
7,vw_retention_strategy_priority:7:chunk_0,retention_strategy,Source: vw_retention_strategy_priority | Topic...


### Result & conclusion

The corpus is now split into overlapping chunks so a query can retrieve focused evidence without sending entire reporting tables or long methodology text to a generator.

In [6]:
embedding_backend = 'tfidf'
embedding_model = None
vector_matrix = None

try:
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer(embedding_model_name)
    vector_matrix = embedding_model.encode(
        chunks_df['chunk_text'].tolist(),
        normalize_embeddings=True,
        show_progress_bar=True
    )
    vector_matrix = np.asarray(vector_matrix, dtype=np.float32)
    embedding_backend = 'sentence_transformers'
except Exception as exc:
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=1
    )
    vector_matrix = vectorizer.fit_transform(chunks_df['chunk_text']).toarray().astype(np.float32)
    import joblib
    joblib.dump(vectorizer, OUTPUT_DIR / 'tfidf_vectorizer.joblib')
    print_result(f'Semantic embeddings were unavailable, so I built a TF-IDF fallback index. Reason: {type(exc).__name__}.')
else:
    print_result(f'Built a semantic embedding matrix with {vector_matrix.shape[0]:,} chunks × {vector_matrix.shape[1]:,} dimensions.')

Result: Semantic embeddings were unavailable, so I built a TF-IDF fallback index. Reason: ModuleNotFoundError.


### Result & conclusion

The notebook prefers semantic embeddings but keeps a reproducible TF-IDF fallback so the retrieval layer can still be built when the embedding model or download is unavailable. The selected backend is recorded explicitly.

In [7]:
np.save(OUTPUT_DIR / 'retrieval_vectors.npy', vector_matrix)

manifest = {
    'backend': embedding_backend,
    'embedding_model': embedding_model_name if embedding_backend == 'sentence_transformers' else None,
    'chunk_size': chunk_size,
    'chunk_overlap': chunk_overlap,
    'chunk_count': int(len(chunks_df)),
    'vector_dimensions': int(vector_matrix.shape[1]),
    'normalized_embeddings': bool(embedding_backend == 'sentence_transformers'),
}
(OUTPUT_DIR / 'retrieval_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print_result(f'Saved the retrieval vectors and manifest using the {embedding_backend} backend.')
print(json.dumps(manifest, indent=2))

Result: Saved the retrieval vectors and manifest using the tfidf backend.
{
  "backend": "tfidf",
  "embedding_model": null,
  "chunk_size": 900,
  "chunk_overlap": 150,
  "chunk_count": 8414,
  "vector_dimensions": 33077,
  "normalized_embeddings": false
}


### Result & conclusion

The retrieval index is now portable: chunk metadata is stored in CSV form and the numerical index plus settings are saved separately. This makes the retrieval step reproducible without rebuilding the corpus.